# LAB 06 — Supervised Learning: Classification

**Module:** Supervised Learning — Classification
**Session:** 06
**Duration:** 150 minutes (10-minute break included)
**Audience:** Başlangıç seviyesi Python / Machine Learning katılımcıları (LAB02, LAB03, LAB04 ve
LAB05'i tamamlamış olmalı)

## LAB02-LAB05'ten Devam

Bu LAB, önceki dört oturumun doğrudan devamıdır:

- **LAB02**'de NumPy, Pandas, filtering/sorting ve groupby/aggregation öğrendiniz.
- **LAB03**'te descriptive statistics, dağılım analizi, outlier analizi ve correlation ile veriyi
  istatistiksel olarak incelediniz.
- **LAB04**'te `SimpleImputer`, `OneHotEncoder`, `StandardScaler`, train/validation/test ayrımı, data
  leakage önleme ve `Pipeline`/`ColumnTransformer` ile tekrar üretilebilir bir preprocessing
  workflow'u inşa ettiniz.
- **LAB05**'te bu preprocessing bilgisini regression modelleriyle (Linear, Ridge, Lasso, Decision
  Tree) birleştirdiniz; baseline karşılaştırması, residual analizi ve model seçimi öğrendiniz.

Bu LAB'da **aynı metodoloji** (baseline → model → değerlendirme → karşılaştırma → seçim → tek
seferlik test) **Classification** problemleri için uygulanacaktır.

## Bu LAB'ın Ana Hedefi

Bu LAB'ın amacı, katılımcıların yalnızca Classification algoritmalarını çalıştırmasını değil,
**güvenilir bir model geliştirme ve değerlendirme sürecini** öğrenmesini sağlamaktır.

## Bu LAB'ın Sonunda Yapabilecekleriniz

1. Classification problemlerini tanımlayabileceksiniz.
2. Binary Classification için uygun target belirleyebileceksiniz.
3. Baseline Classification modeli oluşturabileceksiniz.
4. Logistic Regression eğitebileceksiniz.
5. Decision Tree Classifier geliştirebileceksiniz.
6. Probability Scores ve Decision Threshold kavramlarını açıklayabileceksiniz.
7. Confusion Matrix oluşturup yorumlayabileceksiniz.
8. Accuracy, Precision, Recall ve F1-Score hesaplayabileceksiniz.
9. ROC-AUC ve Precision–Recall analizleri yapabileceksiniz.
10. Class Imbalance problemlerini değerlendirebileceksiniz.
11. Farklı classification modellerini karşılaştırabileceksiniz.
12. Validation verisinde Decision Threshold seçebileceksiniz.
13. Nihai modeli test verisinde bağımsız değerlendirebileceksiniz.
14. Preprocessing ve Classification modellerini tek bir Scikit-learn `Pipeline` içinde
    birleştirebileceksiniz.

## Kullanılan Veri Kümesi: E-Commerce Shipping Data

Bu LAB'da, LAB02-LAB04'te kullandığınız **aynı** E-Commerce Shipping veri kümesi kullanılır:
`data/ecommerce_shipping/Train.csv`. Northwind veri kümesi bu LAB'da **kullanılmaz**.

### İş Problemi: Shipment Delay Classification

**Hedef değişken (target):** `Reached.on.Time_Y.N` — Kaggle dokümantasyonundaki tanıma göre:

- `1` = Ürün zamanında **ulaşmadı** (NOT reached on time)
- `0` = Ürün zamanında **ulaştı** (reached on time)

Bu LAB'da **pozitif sınıf (`1`) = Late Delivery (gecikmiş teslimat)** olarak tanımlanmıştır.

**Amaç:** Bir sevkiyatın zamanında ulaşıp ulaşmayacağını, sevkiyat öncesinde bilinen bilgilerle
tahmin eden bir Binary Classification modeli geliştirmek.

> **Önemli bir eğitim varsayımı — Prediction Time:** Bu LAB, modelin **sevkiyat gönderilmeden önce
> (before shipment dispatch)** tahmin yaptığını varsayar. Bu varsayım altında, `Warehouse_block`,
> `Mode_of_Shipment`, `Cost_of_the_Product`, `Prior_purchases`, `Product_importance`,
> `Discount_offered` ve `Weight_in_gms` alanlarının tahmin anında bilindiği kabul edilir. Bu
> alanların gerçek bir operasyonel sistemde bu zaman diliminde gerçekten erişilebilir olduğu
> **doğrulanmamıştır** — bu LAB boyunca kullanılan bir eğitim varsayımıdır.

> **Önemli sınırlama:** Bu veri kümesi gerçek bir lojistik şirketinin (örn. "Barsan Lojistik")
> verisi **değildir**. Bu LAB'daki modellerin gerçek operasyonel ortamlarda kullanılmaya hazır
> olduğu iddia edilmemektedir.

## LAB İçeriği

| # | Bölüm | Süre |
|---|---|---|
| 1 | Classification Fundamentals & Dataset | 10 dk |
| 2 | Data Splitting & Baseline Models | 15 dk |
| 3 | Logistic Regression | 20 dk |
| 4 | Decision Tree Classification | 20 dk |
| — | **Break** | 10 dk |
| 5 | Confusion Matrix & Classification Metrics | 20 dk |
| 6 | ROC-AUC & Precision–Recall Analysis | 20 dk |
| 7 | Class Imbalance & Decision Threshold | 20 dk |
| 8 | Model Comparison & Final Workshop | 15 dk |

Üç isteğe bağlı **Advanced Challenge**, ana 150 dakikalık programın dışındadır.


---
## Section 1 — Classification Fundamentals & Dataset
**Süre: 10 dakika**

### Learning Objective

Classification probleminin ne olduğunu, Regression'dan (LAB05) farkını ve bu LAB'ın iş problemini
anlamak.

### Concept Explanation

- **Supervised Learning**: Modelin, her gözlem için doğru etiketin (`y`) bilindiği geçmiş verilerden
  öğrenmesidir (LAB05'te de aynı prensip geçerliydi).
- **Classification vs Regression**: Regression (LAB05) sürekli sayısal bir değer tahmin eder.
  Classification, sonlu bir **kategori kümesinden** birini tahmin eder.
- **Binary Classification**: Hedef değişkenin yalnızca **iki** sınıftan oluştuğu classification
  türüdür (bu LAB'da: `0` veya `1`).
- **Features and Target**: `X` (features) girdi değişkenleri, `y` (target) tahmin edilmek istenen
  sınıf etiketidir.
- **Positive and Negative Classes**: Genellikle ilgilenilen/kritik olay **positive class (1)**,
  diğeri **negative class (0)** olarak adlandırılır. Bu LAB'da positive class = Late Delivery.
- **Classification Workflow**: Veri yükleme → feature seçimi → train/validation/test ayrımı →
  preprocessing → model eğitimi → threshold seçimi → değerlendirme → karşılaştırma.
- **Prediction Time**: Bir feature'ın model girdisi olarak kullanılabilmesi için, tahmin yapılacağı
  anda **gerçekten bilinmesi** gerekir.
- **Target Leakage**: Tahmin anında henüz bilinmeyen veya hedefle dolaylı olarak oluşmuş bir bilginin
  feature olarak kullanılmasıdır — yapay olarak yüksek performans gösterir ama gerçek dünyada işe
  yaramaz.

### Hands-On


In [ ]:
# Same environment setup pattern as LAB02-LAB05: resolve the project root, add src/ to sys.path,
# then load the shared configuration and the dataset.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_ecommerce_shipping

config = load_config()
RANDOM_STATE = config["lab06"]["random_seed"]
TARGET = config["lab06"]["target"]
POSITIVE_CLASS = config["lab06"]["positive_class"]

df = load_ecommerce_shipping(config)
print("Dataset shape:", df.shape)
print("Target column:", TARGET, "| Positive class:", POSITIVE_CLASS)
print("Random state (from config/settings.yaml):", RANDOM_STATE)


In [ ]:
print("Columns:", df.columns.tolist())
df.dtypes


In [ ]:
missing_summary = df.isna().sum()
print("Missing value summary (count per column):")
print(missing_summary)
print()
print("Total missing values in the real dataset:", missing_summary.sum())


In [ ]:
print("Target unique values:", sorted(df[TARGET].unique()))
print("Duplicate ID count:", df["ID"].duplicated().sum())


Hedef sütunda eksik değer yok, benzersiz değerler yalnızca `{0, 1}` ve `ID` sütununda tekrarlanan
kayıt yok — bu, dataset'in bu LAB için kullanılabilir olduğunu doğrular.

### Hands-On — Target Distribution


In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_pct = (df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)

print("Target value counts (0 = Reached On Time, 1 = Late Delivery):")
print(target_counts)
print()
print("Target distribution (%):")
print(target_pct)


In [ ]:
import matplotlib.pyplot as plt

# Color palette: fixed categorical/semantic roles reused across the ml-labs notebook series.
BAR_COLOR = "#2a78d6"
HIGHLIGHT_COLOR = "#1baf7a"
WARNING_COLOR = "#eb6834"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["0 (On Time)", "1 (Late)"]
ax.bar(labels, target_counts.values, color=[BAR_COLOR, WARNING_COLOR])
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 50, f"{v}\n({target_pct.values[i]}%)", ha="center")
ax.set_title("Target Distribution — Reached.on.Time_Y.N")
ax.set_ylabel("Order Count")
style_axes(ax)
plt.show()


Pozitif sınıf (`1` — Late Delivery) gözlemlerin **%59.67**'sini, negatif sınıf (`0` — On Time)
**%40.33**'ünü oluşturuyor. Bu, **orta derecede dengesiz (moderately imbalanced)** bir dağılımdır —
şiddetli bir class imbalance (örn. %99/%1) **değildir**. Section 7'de bu konuya geri döneceğiz.

### Hands-On — Feature Selection


**Dahil edilen feature'lar** (tahmin anında bilindiği varsayılan):

| Feature | Tip |
|---|---|
| `Warehouse_block` | Categorical |
| `Mode_of_Shipment` | Categorical |
| `Product_importance` | Categorical |
| `Cost_of_the_Product` | Numerical |
| `Prior_purchases` | Numerical |
| `Discount_offered` | Numerical |
| `Weight_in_gms` | Numerical |

**Dışarıda tutulan sütunlar ve gerekçeleri:**

- `ID` — benzersiz kayıt tanımlayıcısıdır, tahmine yardımcı bir sinyal taşımaz.
- `Reached.on.Time_Y.N` — target değişkenin kendisi.
- `Customer_rating` — müşteri değerlendirmesinin sevkiyat öncesinde mi sonrasında mı oluştuğu
  belirsizdir; **target leakage riski** nedeniyle hariç tutulmuştur.
- `Customer_care_calls` — görüşme sayısının hangi zaman diliminde oluştuğu belirsizdir; tahmin
  anındaki erişilebilirliği doğrulanmadığı için varsayılan modelden çıkarılmıştır.
- `Gender` — bu LAB için açık bir operasyonel gerekçe bulunmamaktadır. Gereksiz demografik
  özelliklerin kullanımı, hem model performansına anlamlı bir katkı sağlamadan karmaşıklık ekler
  hem de **fairness (adillik)** açısından risk oluşturabilir — bu nedenle model girdisi olarak
  kullanılmamıştır.

### Educational Question

**"Sevkiyatın gecikip gecikmeyeceğini tahmin etmek neden bir Classification problemidir?"**

**Beklenen yanıt:** Çünkü hedef değişken, sürekli bir sayı değil, **iki kategorik sınıftan**
(zamanında ulaştı / ulaşmadı) oluşmaktadır.

### Critical Point

Tahmin anında erişilemeyecek bilgileri (örn. sevkiyat sonrası oluşan bilgiler) model girdisi olarak
**kullanmayın** — bu, gerçek dünyada kullanılamayan, yapay olarak iyi performans gösteren bir model
üretir.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: df.groupby("Warehouse_block")[TARGET] üzerinden, her depo bloğu için pozitif sınıf (gecikme)
#       oranını hesaplayın.
# TODO: Depo blokları arasında gözle görülür bir fark var mı?

# print(df.groupby("Warehouse_block")[TARGET].mean().sort_values(ascending=False))


---
## Section 2 — Data Splitting & Baseline Models
**Süre: 15 dakika**

### Learning Objective

Feature matrix ve target vector'ü oluşturmak; veriyi stratified biçimde train/validation/test
kümelerine ayırmak; baseline classification modelinin neden gerekli olduğunu anlamak.

### Concept Explanation

- **Feature Matrix (`X`) / Target Vector (`y`)**: LAB05'teki aynı kavramlar.
- **Train / Validation / Test**: LAB04/LAB05'teki aynı üçlü ayrım — training model öğrenimi,
  validation model/threshold seçimi, test ise seçilenin **tek seferlik** bağımsız değerlendirmesi
  içindir.
- **Stratified Splitting**: Bölünen her alt kümede (`train`/`val`/`test`), sınıf oranlarının
  (`%59.67` / `%40.33`) **orijinal dağılıma yakın** kalmasını sağlayan split yöntemidir.
  Classification'da, LAB05'teki kronolojik regresyon split'inin aksine, stratified split
  tercih edilir.
- **Random State**: Tekrar üretilebilirlik için sabitlenen rastgelelik tohumu.
- **Baseline Classification / `DummyClassifier`**: Herhangi bir feature kullanmadan basit bir
  kurala (örn. her zaman en sık görülen sınıfı tahmin et) dayanan referans model.

### Hands-On — Feature Matrix & Target Vector


In [ ]:
numerical_features = ["Cost_of_the_Product", "Prior_purchases", "Discount_offered", "Weight_in_gms"]
categorical_features = ["Warehouse_block", "Mode_of_Shipment", "Product_importance"]
feature_columns = numerical_features + categorical_features

X = df[feature_columns]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)


### Data Splitting — Stratified Two-Stage Split

LAB05'te kronolojik split kullanmıştık (zaman serisi niteliğindeki Northwind verisi için). Bu
dataset'te güvenilir bir zaman sırası bilgisi yok, bu yüzden **stratified random split**
kullanacağız: önce `%70` training / `%30` geri kalan, ardından geri kalanı `%50`/`%50` ile
validation/test olarak ikiye bölerek nihai `%70/%15/%15` oranını elde edeceğiz.
`stratify=y` parametresi, her alt kümede pozitif sınıf oranının orijinal dağılıma yakın kalmasını
sağlar.

### Hands-On


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)


In [ ]:
# Validation: split sizes, class distributions, index overlap, reproducibility.
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} | n={len(yy):5d} | positive rate: {yy.mean():.4f}")

print()
print("Index overlap train/val :", len(set(X_train.index) & set(X_val.index)))
print("Index overlap train/test:", len(set(X_train.index) & set(X_test.index)))
print("Index overlap val/test  :", len(set(X_val.index) & set(X_test.index)))


Üç kümede de pozitif sınıf oranı (`~0.597`) orijinal dağılıma çok yakın — stratified split başarılı
oldu. Kümeler arasında index çakışması yok (`0`), yani her gözlem tam olarak bir kümeye ait. Aynı
`random_state` ile her çalıştırmada aynı bölünme elde edilir (reproducibility).

### Hands-On — Baseline Model


In [ ]:
from sklearn.dummy import DummyClassifier

baseline_pipe = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline always predicts class:", baseline_pipe.predict(X_val.iloc[:5]))
print("classes_:", baseline_pipe.classes_)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pred_val_baseline = baseline_pipe.predict(X_val)

print(f"Accuracy : {accuracy_score(y_val, pred_val_baseline):.4f}")
print(f"Precision: {precision_score(y_val, pred_val_baseline, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_val, pred_val_baseline, zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_val, pred_val_baseline, zero_division=0):.4f}")


### Interpretation

`DummyClassifier(strategy="most_frequent")`, validation setindeki **en sık görülen sınıfı** (`1` —
Late) her zaman tahmin eder. Bu dataset'te pozitif sınıf zaten çoğunlukta (`%59.67`) olduğu için bu
baseline:

- **Accuracy ≈ %59.70** elde eder — rastgele tahminden (%50) daha iyi görünür, ama bu yalnızca
  sınıf oranını yansıtır, gerçek bir ayırt etme (discrimination) yeteneği göstermez.
- Pozitif sınıf için **Recall = 1.0** (her zaman `1` dediği için tüm gerçek `1`'leri "yakalar"), ama
  bu önemsizdir çünkü negatif sınıfı (`0` — On Time) **hiçbir zaman** tahmin etmez.

**Bu dataset'te "yüksek accuracy ama pozitif sınıfı tamamen gözden kaçırma" klasik senaryosu
gerçekleşmiyor** — çünkü positive class burada zaten çoğunluk sınıfı. Burada gözlemlediğimiz, farklı
ama ilişkili bir ders: **Accuracy tek başına, bir modelin gerçek bir ayırt etme yeteneğine sahip
olup olmadığını göstermez.** Section 7'de, pozitif sınıfın azınlıkta olduğu bir senaryoyu (sentetik
bir örnekle) göstererek "Accuracy Paradox"un klasik formunu da inceleyeceğiz.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: DummyClassifier(strategy="prior") ile ikinci bir baseline oluşturun.
# TODO: predict() çıktılarının most_frequent ile aynı olup olmadığını, ancak predict_proba()
#       çıktılarının nasıl farklılaştığını karşılaştırın.

# baseline_prior = DummyClassifier(strategy="prior").fit(X_train, y_train)
# print(baseline_prior.predict_proba(X_val.iloc[:3]))
# print(baseline_pipe.predict_proba(X_val.iloc[:3]))


---
## Section 3 — Logistic Regression
**Süre: 20 dakika**

### Learning Objective

Logistic Regression'ın matematiğini (sigmoid, log-odds) ve Scikit-learn uygulamasını öğrenmek;
`predict()` ile `predict_proba()` arasındaki farkı anlamak.

### Mathematical Explanation

Linear Score (LAB05'teki `z` ile aynı mantık):

$$z=\beta_0+\sum_{j=1}^{p}\beta_jx_j$$

Sigmoid Function (lineer skoru `[0, 1]` aralığına sıkıştırır):

$$\sigma(z)=\frac{1}{1+e^{-z}}$$

Probability Estimation:

$$P(y=1\mid X)=\sigma(z)$$

Logistic Regression, LAB05'teki Linear Regression'ın aksine bir sayı değil, **sınıf 1 olma
olasılığını** tahmin eder.

### Hands-On — Preprocessing Pipeline

LAB04/LAB05'te öğrendiğiniz preprocessing bileşenlerini, bu LAB'ın feature şemasına göre yeniden
kuruyoruz (LAB04/LAB05'teki fit edilmiş pipeline'lar farklı veri şemaları içindir, burada
kullanılamaz).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)


### Critical Engineering Rule

Preprocessing ve estimator **aynı `Pipeline` içinde** bulunmalıdır — bu, leakage'ı önler (pipeline
yalnızca `.fit()` çağrıldığı veride öğrenir) ve kodu tekrar üretilebilir kılar.

### Hands-On — Training


In [ ]:
import warnings
from sklearn.linear_model import LogisticRegression

logreg_pipe = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(random_state=RANDOM_STATE)),
])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    logreg_pipe.fit(X_train, y_train)
    if caught:
        print(f"{len(caught)} warning(s) raised -> {[str(w.message) for w in caught]}")
    else:
        print("No warnings raised during fit.")


Bu çalıştırmada hiçbir `ConvergenceWarning` oluşmadı — varsayılan `max_iter` (100) bu dataset için
yeterli. Eğer bu uyarı oluşsaydı, yalnızca gizlemek yerine genellikle `max_iter` değerini artırmak
veya feature'ların doğru ölçeklendiğini (pipeline'daki `StandardScaler`) kontrol etmek doğru çözüm
olurdu.

### Hands-On — `predict()` vs `predict_proba()`


In [ ]:
pred_val_hard = logreg_pipe.predict(X_val)
pred_val_proba = logreg_pipe.predict_proba(X_val)

print("classes_:", logreg_pipe.named_steps["model"].classes_)
print()
print("predict() -> hard class labels, first 5:", pred_val_hard[:5])
print()
print("predict_proba() -> probability per class, first 5 rows:")
print(pred_val_proba[:5])


`classes_` dizisi `[0, 1]` olduğu için, `predict_proba()`'nın **ikinci sütunu** (`[:, 1]`) sınıf
`1`'in (Late) olasılığıdır. Bu sırayı asla varsaymayın — her zaman `classes_` ile doğrulayın.

### Hands-On — Evaluation at Default Threshold (0.5)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba_val_lr = logreg_pipe.predict_proba(X_val)[:, 1]
pred_train_lr = logreg_pipe.predict(X_train)

print(f"Train Accuracy: {accuracy_score(y_train, pred_train_lr):.4f}")
print(f"Val Accuracy  : {accuracy_score(y_val, pred_val_hard):.4f}")
print(f"Val Precision : {precision_score(y_val, pred_val_hard):.4f}")
print(f"Val Recall    : {recall_score(y_val, pred_val_hard):.4f}")
print(f"Val F1-Score  : {f1_score(y_val, pred_val_hard):.4f}")


Baseline'a (Val Accuracy ≈ %59.70) kıyasla, Logistic Regression Val Accuracy'yi ≈ %64.48'e
yükseltti — mütevazı ama gerçek bir iyileşme. Decision Threshold varsayılan olarak **0.5**'tir:
`predict_proba()[:, 1] >= 0.5` ise sınıf `1` tahmin edilir. **Bu aşamada threshold optimizasyonu
yapmıyoruz** — bunu Section 7'de ele alacağız.

### Hands-On — Coefficient Interpretation


In [ ]:
feature_names = logreg_pipe.named_steps["preprocessing"].get_feature_names_out()
coefficients = pd.Series(logreg_pipe.named_steps["model"].coef_[0], index=feature_names)

print("Intercept:", logreg_pipe.named_steps["model"].intercept_[0])
print()
print("Top 10 coefficients by absolute magnitude:")
print(coefficients.reindex(coefficients.abs().sort_values(ascending=False).index).head(10))


In [ ]:
# Cross-check: does the raw (unscaled) correlation between the numerical features and the target
# agree with the direction of the Logistic Regression coefficients above?
print(df[numerical_features + [TARGET]].corr()[TARGET].drop(TARGET))


En büyük katsayı `Discount_offered` (+1.83) — daha yüksek indirim sunulan siparişlerin gecikme
olasılığı daha yüksek görünüyor. Bu yön, ham (ölçeklenmemiş) korelasyon analiziyle de tutarlı:
`Discount_offered` ile target arasında +0.40 korelasyon var. `Weight_in_gms` negatif (-0.33) —
daha ağır paketlerin gecikme olasılığı biraz daha düşük; bu da ham korelasyonla (-0.27) aynı
yöndedir.

### Technical Notes

- **Probability score ≠ kesin gerçekleşme olasılığı.** Bu olasılıklar **calibration** (kalibrasyon)
  olmadan, gerçek dünya frekanslarıyla kusursuz örtüştüğü iddia edilemez.
- **Katsayılar nedensel (causal) bir etki değildir** — yalnızca bu veri kümesindeki gözlemsel
  ilişkiyi yansıtır.
- **Scaling ve regularization**: `LogisticRegression`'ın varsayılan L2 regularization'ı
  (`penalty="l2"`), LAB05'teki Ridge'e benzer şekilde feature ölçeğine duyarlıdır — bu nedenle
  `StandardScaler` burada da önemlidir.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: coefficients Series'inde en küçük (sıfıra en yakın) 5 katsayıyı yazdırın.
# TODO: Bu feature'ların Late Delivery tahmini üzerinde en az etkiye sahip görünenler olduğunu
#       yorumlayın.

# print(coefficients.reindex(coefficients.abs().sort_values(ascending=True).index).head(5))


---
## Section 4 — Decision Tree Classification
**Süre: 20 dakika**

### Learning Objective

Decision Tree Classification'ın temel mantığını (Gini impurity, splitting) öğrenmek ve `max_depth`
ile overfitting arasındaki ilişkiyi gözlemlemek.

### Concept Explanation

- **Decision Tree Fundamentals**: LAB05'teki Decision Tree Regression ile aynı yapı — veri, feature
  değerlerine göre art arda ikiye bölünür (splitting).
- **Classification Trees**: Her **leaf node** (yaprak), o yaprağa düşen training örneklerinin
  **çoğunluk sınıfını** (ve sınıf oranlarını, `predict_proba` için) döndürür.
- **Gini Impurity**: Bir düğümdeki sınıf karışıklığını ölçer: $Gini = 1 - \sum_k p_k^2$. `0` =
  tamamen saf (tek sınıf), `0.5` = iki sınıfın eşit karışımı (binary classification'da maksimum).
- **Entropy**: Alternatif bir karışıklık ölçütü: $Entropy = -\sum_k p_k \log_2(p_k)$.
- **Information Gain**: Bir split'in, ebeveyn düğüme kıyasla karışıklığı ne kadar azalttığıdır.
  Scikit-learn, her split'te bu azalmayı en çoklayan feature/eşik çiftini seçer.
- **Tree Depth / Overfitting**: LAB05'te gördüğünüz gibi, derinlik arttıkça model training verisini
  ezberlemeye başlayabilir.

### Hands-On — A Small Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

small_tree_pipe = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
])
small_tree_pipe.fit(X_train, y_train)

pred_val_tree = small_tree_pipe.predict(X_val)
print(f"Val Accuracy : {accuracy_score(y_val, pred_val_tree):.4f}")
print(f"Val Precision: {precision_score(y_val, pred_val_tree):.4f}")
print(f"Val Recall   : {recall_score(y_val, pred_val_tree):.4f}")


### Hands-On — Comparing Tree Depths


In [ ]:
tree_results = []
tree_models = {}

for depth in [2, 3, 5, None]:
    pipe = Pipeline(steps=[("preprocessing", preprocessor), ("model", DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    tree_models[depth] = pipe

    pred_train_tree = pipe.predict(X_train)
    pred_val_tree = pipe.predict(X_val)
    tree_results.append({
        "max_depth": depth,
        "Train_Accuracy": accuracy_score(y_train, pred_train_tree),
        "Val_Accuracy": accuracy_score(y_val, pred_val_tree),
        "Val_Precision": precision_score(y_val, pred_val_tree),
        "Val_Recall": recall_score(y_val, pred_val_tree),
        "Val_F1": f1_score(y_val, pred_val_tree),
    })

tree_results_df = pd.DataFrame(tree_results)
tree_results_df


### Interpretation

- `max_depth=2`: En iyi genel validation performansı (Accuracy ≈ %68.6, F1 ≈ 0.70), train/val farkı
  küçük — iyi bir genelleme.
- `max_depth=3` ve `max_depth=5`: **Precision çok yüksek** (≈ %98 ve ≈ %90) ama **Recall düşük**
  (≈ %45-49) — model, yalnızca çok "net" durumlarda `1` tahmin ediyor, çoğu gecikmeyi kaçırıyor.
- `max_depth=None` (sınırsız derinlik): **Train Accuracy = 1.0** (modelin training verisini
  ezberlediğinin kanıtı) ama Val Accuracy (≈ %64.1) `max_depth=2`'den **daha düşük** — klasik bir
  **overfitting** örneği.

### Hands-On — Visualizing a Small Tree


In [ ]:
from sklearn.tree import plot_tree

viz_tree_model = tree_models[2].named_steps["model"]
viz_tree_feature_names = tree_models[2].named_steps["preprocessing"].get_feature_names_out()

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(
    viz_tree_model,
    feature_names=viz_tree_feature_names,
    class_names=["On Time", "Late"],
    filled=True,
    fontsize=8,
    ax=ax,
)
plt.show()


Not: Yalnızca `max_depth=2` olan küçük ağacı görselleştirdik — `max_depth=None` gibi çok büyük bir
ağacı tamamen çizdirmek okunaksız olur ve bu LAB'ın kapsamı dışındadır.

### Technical Discussion

- **Ağaç derinliği arttıkça model neden overfit olabilir?** Her ek derinlik, training verisindeki
  daha küçük (ve genellikle daha gürültülü) alt gruplara özel kurallar öğrenme fırsatı verir.
- **Decision Tree scaling gerektirir mi?** Hayır — bölme kararları feature'ın mutlak ölçeğinden
  değil sıralamasından etkilenir. Ancak burada, Logistic Regression ile **adil bir karşılaştırma**
  için aynı `preprocessor` (dolayısıyla ölçeklenmiş feature'lar) kullanıldı; bu Decision Tree için
  gerekli olmasa da sonucu yanlış yönde etkilemez.
- **Logistic Regression ve Decision Tree hangi yönlerden farklıdır?** Logistic Regression doğrusal
  bir karar sınırı öğrenir ve katsayılarla yorumlanır; Decision Tree doğrusal olmayan,
  dikdörtgen-şeklinde bölgeler öğrenir ve kurallarla (if/else) yorumlanır.

### Important

Farklı model aileleri için farklı preprocessing kullanılması mümkündür (örn. Decision Tree için
scaling atlanabilir), ancak karşılaştırmaların **tutarlı ve açıkça belgelenmiş** olması gerekir —
bu LAB'da bu tutarlılık, tüm modeller için aynı `preprocessor` kullanılarak sağlanmıştır.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: tree_results_df tablosuna, Train_Accuracy ile Val_Accuracy arasındaki farkı yeni bir
#       sütun olarak ekleyin.
# TODO: Bu farkın max_depth ile nasıl değiştiğini yorumlayın (büyüyen fark = artan overfitting
#       sinyali — LAB05 Section 5'teki Train/Val Gap mantığının aynısı).

# tree_results_df["Train_Val_Gap"] = tree_results_df["Train_Accuracy"] - tree_results_df["Val_Accuracy"]
# print(tree_results_df)


---
## ☕ Break — 10 dakika

Kısa bir ara verelim. Devam ederken: `preprocessor`, `logreg_pipe`, `tree_models`,
`X_train`/`X_val`/`X_test` ve `y_train`/`y_val`/`y_test` değişkenleri bundan sonraki tüm bölümlerde
kullanılacaktır.


---
## Section 5 — Confusion Matrix & Classification Metrics
**Süre: 20 dakika**

### Learning Objective

Confusion Matrix'in dört bileşenini (TP/TN/FP/FN) ve bunlardan hesaplanan Accuracy, Precision,
Recall, F1-Score metriklerini anlamak ve iş bağlamında yorumlamak.

### Concept Explanation

- **True Positive (TP)**: Gerçekte `1` (Late) olan ve model tarafından `1` tahmin edilen.
- **True Negative (TN)**: Gerçekte `0` (On Time) olan ve model tarafından `0` tahmin edilen.
- **False Positive (FP)**: Gerçekte `0` olan ama model tarafından `1` tahmin edilen.
- **False Negative (FN)**: Gerçekte `1` olan ama model tarafından `0` tahmin edilen.

### Mathematical Explanations

$$Accuracy=\frac{TP+TN}{TP+TN+FP+FN}$$

$$Precision=\frac{TP}{TP+FP}$$

$$Recall=\frac{TP}{TP+FN}$$

$$F_1=2\cdot\frac{Precision\cdot Recall}{Precision+Recall}$$

### Hands-On — Confusion Matrix


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# labels=[0, 1] fixes the row/column order explicitly: 0 = On Time, 1 = Late.
cm_lr = confusion_matrix(y_val, pred_val_hard, labels=[0, 1])
print(cm_lr)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm_lr, display_labels=["On Time (0)", "Late (1)"]).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — LogisticRegression (Validation, threshold=0.5)")
plt.show()


In [ ]:
tn, fp, fn, tp = cm_lr.ravel()
print(f"TN={tn}  FP={fp}")
print(f"FN={fn}  TP={tp}")


### Hands-On — Metrics & Classification Report


In [ ]:
from sklearn.metrics import classification_report

print(f"Accuracy : {accuracy_score(y_val, pred_val_hard):.4f}")
print(f"Precision: {precision_score(y_val, pred_val_hard):.4f}")
print(f"Recall   : {recall_score(y_val, pred_val_hard):.4f}")
print(f"F1-Score : {f1_score(y_val, pred_val_hard):.4f}")
print()
print(classification_report(y_val, pred_val_hard, target_names=["On Time (0)", "Late (1)"]))


### Business Interpretation

- **False Positive** (burada: `FP=259`): Zamanında teslim edilecek bir sevkiyata **gecikme alarmı**
  verilmesi — gereksiz bir operasyonel müdahale (örn. ekstra kargo takibi, müşteriye önceden
  "gecikebilir" bildirimi) tetiklenebilir.
- **False Negative** (burada: `FN=327`): Gerçekte gecikecek bir sevkiyatın **zamanında ulaşacak**
  şekilde tahmin edilmesi — potansiyel olarak daha maliyetli bir hata, çünkü önlem alınmaz.

### Discussion

**Hangi hata türü daha maliyetlidir?** Gerçek iş maliyetleri (müşteri kaybı, operasyonel müdahale
maliyeti vb.) bu dataset'te bilinmediği için, bu soruya **uydurma parasal değerler** atayarak kesin
bir cevap vermiyoruz. Ancak genel olarak, "önlenebilir bir gecikmeyi kaçırmak" (FN) genellikle
"gereksiz bir uyarı vermek"ten (FP) daha maliyetli kabul edilir — bu, Section 7'de threshold
seçimini etkileyecek bir tartışmadır.

### Student Exercise

Confusion Matrix üzerinden Precision ve Recall değerlerini **elle** hesaplayıp Scikit-learn
sonuçlarıyla karşılaştırın.


In [ ]:
# Student Exercise:
# TODO: Yukarıdaki tn, fp, fn, tp değerlerini kullanarak Precision ve Recall'u elle (manuel) hesaplayın.
# TODO: Bu değerleri precision_score/recall_score çıktılarıyla karşılaştırın (np.isclose kullanabilirsiniz).

# manual_precision = tp / (tp + fp)
# manual_recall = tp / (tp + fn)
# print(manual_precision, manual_recall)
# print(np.isclose(manual_precision, precision_score(y_val, pred_val_hard)))


---
## Section 6 — ROC-AUC & Precision–Recall Analysis
**Süre: 20 dakika**

### Learning Objective

Threshold'dan bağımsız (threshold-independent) sıralama (ranking) metriklerini — ROC-AUC ve
Average Precision — anlamak ve ne zaman hangisinin daha bilgilendirici olduğunu değerlendirmek.

### Concept Explanation

- **Classification Scores**: `predict_proba()`'nın ürettiği sürekli olasılık skorları (Section
  3'te gördüğünüz).
- **Threshold-Independent Ranking Metrics**: Accuracy/Precision/Recall/F1, **tek bir threshold**'a
  (Section 3'te 0.5) bağlıdır. ROC-AUC ve Average Precision, modelin **tüm olası threshold'lar**
  için ne kadar iyi sıraladığını özetler.
- **True Positive Rate (TPR)** = Recall = $\frac{TP}{TP+FN}$
- **False Positive Rate (FPR)** = $\frac{FP}{FP+TN}$
- **ROC Curve**: Her threshold için (FPR, TPR) noktalarını çizer.
- **ROC-AUC**: ROC eğrisinin altında kalan alan; `0.5` = rastgele tahmin, `1.0` = kusursuz ayırt
  etme.
- **Precision–Recall Curve**: Her threshold için (Recall, Precision) noktalarını çizer.
- **Average Precision (AP)**: PR eğrisinin özet istatistiği.
- **Class Prevalence**: Pozitif sınıfın veri kümesindeki oranı (bu LAB'da ≈ %59.67) — AP'nin
  yorumlanmasında önemlidir (bkz. aşağıda).

### Hands-On — ROC Curve & Precision–Recall Curve


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

fpr, tpr, roc_thresholds = roc_curve(y_val, proba_val_lr)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_val, proba_val_lr)

roc_auc = roc_auc_score(y_val, proba_val_lr)
ap = average_precision_score(y_val, proba_val_lr)

print(f"Val ROC-AUC            : {roc_auc:.4f}")
print(f"Val Average Precision  : {ap:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color=BAR_COLOR, linewidth=2, label=f"LogisticRegression (AUC={roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], color=GRID_COLOR, linewidth=2, linestyle="--", label="Random (AUC=0.5)")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(frameon=False)
style_axes(axes[0])

axes[1].plot(recall_curve, precision_curve, color=BAR_COLOR, linewidth=2, label=f"LogisticRegression (AP={ap:.3f})")
axes[1].axhline(y.mean(), color=GRID_COLOR, linewidth=2, linestyle="--", label=f"Class prevalence ({y.mean():.3f})")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(frameon=False)
style_axes(axes[1])

plt.tight_layout()
plt.show()


### Important

Bu grafikler **positive class (`1`) skorlarıyla** (`predict_proba()[:, 1]`) oluşturuldu — `predict()`
çıktısı (hard label) **değil**, çünkü ROC-AUC/AP, modelin tüm threshold'lar boyunca sıralama
kalitesini ölçer; sabit bir threshold'a indirgenmiş hard label'lar bu bilgiyi kaybeder.

### Precision–Recall Analysis

PR Curve, özellikle **pozitif sınıf nadir olduğunda** ROC Curve'den daha bilgilendiricidir — çünkü
ROC-AUC, büyük bir negatif sınıf içinde küçük sayıda False Positive'in etkisini "sulandırabilir".
Bu LAB'daki dataset'te pozitif sınıf zaten çoğunlukta (%59.67) olduğu için bu özellikle kritik bir
senaryo değil, ancak teknik olarak her iki grafiği birlikte incelemek her zaman daha güvenlidir.

### Average Precision

`average_precision_score()`, Precision-Recall eğrisinin altındaki alanı **trapezoidal integrasyon**
ile değil, **recall değişimlerinde precision'ın ağırlıklı ortalaması** olarak hesaplar — bu iki
yöntem matematiksel olarak **aynı değildir** (AP, trapezoidal PR-AUC'ye göre genellikle daha
"iyimser olmayan", kesikli (step-function) bir yaklaşımdır).

Dikkat edin: **AP (≈ 0.84), class prevalence'tan (≈ 0.597) önemli ölçüde yüksek** — bu, modelin
rastgele bir sıralamadan gerçekten daha iyi olduğunu gösterir. AP'yi her zaman class prevalence'a
göre yorumlayın; AP'nin kendisi prevalence'tan bağımsız bir "mutlak" iyilik ölçütü değildir.

### Critical Rules

- ROC-AUC'nin yüksek olması, modelin **operasyonel başarısını garanti etmez** — hâlâ doğru bir
  decision threshold seçilmesi gerekir (Section 7).
- AP değerini class prevalence'dan **bağımsız** yorumlamayın.
- Threshold seçimini **test verisi üzerinden yapmayın** (Section 7'de validation kullanacağız).
- **Calibration** (olasılıkların gerçek frekanslarla örtüşmesi) ile **Ranking Performance**
  (ROC-AUC/AP'nin ölçtüğü sıralama kalitesi) birbirine karıştırılmaması gereken farklı kavramlardır.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: tree_models[2] (Section 4'teki max_depth=2 Decision Tree) için Val ROC-AUC ve Val AP değerlerini
#       hesaplayın.
# TODO: Bu değerleri LogisticRegression'ın ROC-AUC/AP değerleriyle karşılaştırın.

# proba_val_tree = tree_models[2].predict_proba(X_val)[:, 1]
# print(roc_auc_score(y_val, proba_val_tree), average_precision_score(y_val, proba_val_tree))


---
## Section 7 — Class Imbalance & Decision Threshold
**Süre: 20 dakika**

Bu LAB'ın en kritik bölümlerinden biridir.

### Part A — Class Imbalance

### Learning Objective

Class Imbalance kavramını, "Accuracy Paradox"u ve bunun bu LAB'ın gerçek dataset'inde neden
**şiddetli biçimde** gözlemlenmediğini anlamak.

### Concept Explanation

- **Majority / Minority Class**: Veri kümesinde daha çok / daha az gözlenen sınıf.
- **Class Distribution**: Section 1'de hesapladığımız gibi, bu dataset'te `%59.67` / `%40.33`.
- **Accuracy Paradox**: Sınıflar dengesizken, bir modelin yüksek Accuracy'ye sahip olmasının,
  azınlık sınıfı tespit etme yeteneğiyle **hiçbir ilgisi olmayabileceği** durumdur.
- **False Negative Costs**: Bazı iş bağlamlarında (örn. hastalık teşhisi), bir pozitif örneği
  kaçırmanın (FN) maliyeti bir False Positive'den çok daha yüksektir.
- **Precision–Recall Trade-off**: Section 3B'de somut olarak göreceğiz.
- **Class Weights / Resampling**: Imbalance'ı ele almak için kullanılan teknikler — bu LAB'ın ana
  akışında **uygulanmaz** (Advanced Challenge 1'de class weight'i isteğe bağlı olarak
  inceleyeceksiniz), çünkü bu dataset şiddetli bir imbalance göstermiyor.

### Hands-On — Real Dataset Class Balance


In [ ]:
positive_count = int((y == 1).sum())
negative_count = int((y == 0).sum())
positive_rate = y.mean()
majority_rate = max(positive_rate, 1 - positive_rate)

print(f"Positive Count (Late)   : {positive_count}")
print(f"Negative Count (On Time): {negative_count}")
print(f"Positive Rate           : {positive_rate:.4f}")
print(f"Majority Class Rate     : {majority_rate:.4f}")


### Interpretation

Bu dataset'te sınıf oranı `%59.67 / %40.33` — bu **orta derecede dengesiz (moderately imbalanced)**
bir dağılımdır, ancak **şiddetli (severe)** bir imbalance değildir (örn. %99/%1 gibi). Bu nedenle:

- Section 2'deki `DummyClassifier(most_frequent)` baseline'ı, `%59.70` Accuracy elde etti — bu
  "yüksek" bir Accuracy değildir (rastgele tahminden sadece biraz daha iyi), bu yüzden klasik
  "yüksek Accuracy ile yanıltıcı yüksek performans algısı" senaryosu burada **net biçimde
  gözlemlenmiyor**.
- Yine de, bu baseline **negatif sınıfı (`0`) hiç tespit edemiyor** (Recall(0) = 0) — bu, dengeli
  olmayan dağılımların, basit baseline'ları yanıltıcı biçimde "makul" gösterebileceğinin gerçek bir
  örneğidir.

### Educational Demonstration — [SYNTHETIC] Severe Imbalance

Gerçek dataset'teki imbalance şiddetli olmadığı için, "Accuracy Paradox"un klasik, çarpıcı formunu
göstermek üzere **küçük, açıkça etiketlenmiş bir synthetic örnek** kullanacağız. **Bu örnek
E-Commerce Shipping veri kümesinden değildir ve gerçek bir dataset istatistiği olarak
sunulmamaktadır.**


In [ ]:
# [SYNTHETIC] Educational example only — NOT derived from the E-Commerce Shipping dataset.
synthetic_negative_count = 990
synthetic_positive_count = 10

synthetic_y_true = np.array([0] * synthetic_negative_count + [1] * synthetic_positive_count)
synthetic_y_pred_always_negative = np.zeros_like(synthetic_y_true)  # a model that always predicts 0

synthetic_accuracy = accuracy_score(synthetic_y_true, synthetic_y_pred_always_negative)
synthetic_recall_positive = recall_score(synthetic_y_true, synthetic_y_pred_always_negative, zero_division=0)

print(f"[SYNTHETIC] n = {len(synthetic_y_true)} (990 Negative, 10 Positive)")
print(f"[SYNTHETIC] A model that ALWAYS predicts 0 (Negative):")
print(f"[SYNTHETIC]   Accuracy             = {synthetic_accuracy:.4f}")
print(f"[SYNTHETIC]   Positive Class Recall = {synthetic_recall_positive:.4f}")


**[SYNTHETIC] Yorum:** Her kaydı "Negative" tahmin eden bir model, `%99` Accuracy elde ediyor —
kulağa çok iyi geliyor — ama pozitif sınıfı (örn. nadir bir hastalık, bir dolandırıcılık işlemi)
**hiçbir zaman** yakalayamıyor (Recall = 0). Bu, Accuracy Paradox'un klasik formudur: **şiddetli
imbalance altında, Accuracy tek başına tamamen yanıltıcı olabilir.** Bu LAB'ın gerçek dataset'inde
imbalance bu kadar şiddetli olmadığı için bu etki bu derece çarpıcı gözlemlenmez — ama prensip
genel geçerlidir ve şiddetli dengesiz dataset'lerle çalışırken akılda tutulmalıdır.

---

### Part B — Decision Threshold

### Learning Objective

Decision Threshold'un Precision/Recall üzerindeki etkisini somut sayılarla gözlemlemek ve
validation verisi üzerinde, önceden tanımlı bir hedefe göre bir threshold seçmek.

### Concept Explanation

- **Probability Threshold**: `predict_proba()[:, 1] >= threshold` kuralıyla hard label üretme
  sınırı. Section 3'te varsayılan `0.5`'i kullandık.
- **Precision–Recall Trade-off**: Threshold düşürüldüğünde model daha fazla `1` tahmin eder — Recall
  artar, genellikle Precision düşer (ve tersi).
- **Threshold Optimization**: Belirli bir hedefe (örn. maksimum F1, minimum Precision koşulu)
  göre threshold seçme süreci.
- **Business Costs**: Gerçek bir uygulamada threshold, FP ve FN'in gerçek maliyetlerine göre
  seçilir — bu LAB'da bu maliyetler bilinmediği için **teknik** bir hedef kullanacağız.

### Hands-On — Comparing Thresholds


In [ ]:
def evaluate_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "threshold": threshold,
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "FP": fp,
        "FN": fn,
    }


threshold_comparison = pd.DataFrame([
    evaluate_threshold(y_val, proba_val_lr, t) for t in [0.30, 0.50, 0.70]
])
threshold_comparison


Threshold düşürüldükçe (0.70 → 0.30): **Recall artıyor** (0.39 → 0.98, yani model gecikmeleri daha
az kaçırıyor), ama **False Positive sayısı da artıyor** (6 → 641, yani model çok daha fazla
zamanında-teslim siparişine "gecikecek" diyor). Bu, somut sayılarla gösterilmiş bir
Precision–Recall trade-off'udur.

### Hands-On — Threshold vs Metrics Visualization


In [ ]:
threshold_sweep = np.arange(0.05, 0.96, 0.01)
sweep_results = pd.DataFrame([evaluate_threshold(y_val, proba_val_lr, t) for t in threshold_sweep])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_results["threshold"], sweep_results["Precision"], color=BAR_COLOR, linewidth=2, label="Precision")
ax.plot(sweep_results["threshold"], sweep_results["Recall"], color=WARNING_COLOR, linewidth=2, label="Recall")
ax.plot(sweep_results["threshold"], sweep_results["F1"], color=HIGHLIGHT_COLOR, linewidth=2, label="F1")
ax.axvline(0.5, color=GRID_COLOR, linewidth=1.5, linestyle="--", label="Default (0.5)")
ax.set_title("Decision Threshold vs Precision / Recall / F1 (Validation)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend(frameon=False)
style_axes(ax)
plt.show()


### Threshold Selection

**Yaklaşım A — "Validation F1-Score'u en yüksek yapan threshold'u seç":** Bu, sık kullanılan bir
yaklaşımdır. Uygulayalım ve sonucu dikkatle inceleyelim.


In [ ]:
best_f1_row = sweep_results.loc[sweep_results["F1"].idxmax()]
print("Approach A — threshold that maximizes F1:")
print(best_f1_row)


**Dikkat:** Bu yaklaşım, threshold'u **çok düşük** bir değere (`~0.05-0.20`) çekiyor — bu da fiilen
"her zaman `1` tahmin et" davranışına (Section 2'deki baseline'a) yakınsıyor (Precision ≈ 0.597,
Recall ≈ 1.0). **Neden?** Çünkü bu dataset'te positive class zaten **çoğunluk sınıfı** (`%59.67`).
F1-Score, Precision ve Recall'un harmonik ortalamasıdır; positive class çoğunluktayken, threshold'u
düşürüp daha fazla `1` tahmin etmek Recall'u hızla 1.0'a yaklaştırırken Precision'ı yalnızca class
prevalence'a kadar düşürür — bu da ham F1'i yapay olarak yüksek gösterir. **Bu, saf F1
maksimizasyonunun, positive class çoğunluktayken dejenere (anlamsız) bir sonuca yakınsayabileceğinin
somut bir örneğidir.**

**Yaklaşım B — "Belirli bir minimum Precision koşulunu sağlayarak Recall'u maksimize et":** Bu LAB
için daha anlamlı bir hedef seçelim: **"En az %75 Precision sağlarken Recall'u maksimize et."** Bu,
"gecikme alarmı verdiğimizde en az %75 oranında doğru olalım" şeklinde bir iş tercihini temsil eder.


In [ ]:
MIN_PRECISION = 0.75

eligible = sweep_results[sweep_results["Precision"] >= MIN_PRECISION]
best_constrained_row = eligible.loc[eligible["Recall"].idxmax()]

CHOSEN_THRESHOLD = round(float(best_constrained_row["threshold"]), 2)
print("Approach B — max Recall subject to Precision >= 0.75:")
print(best_constrained_row)
print()
print("Chosen threshold:", CHOSEN_THRESHOLD)


Bu yaklaşım, `threshold ≈ 0.53`'te **Precision ≈ 0.75, Recall ≈ 0.60** (F1 ≈ 0.67) veren, dejenere
olmayan, anlamlı bir çalışma noktası veriyor. Varsayılan `0.5`'e kıyasla biraz daha yüksek Precision
(önlem aldığımızda daha güvenilir olalım) karşılığında biraz daha düşük Recall kabul ediyoruz. Bu
LAB'ın kalan bölümlerinde **bu threshold'u** (`CHOSEN_THRESHOLD`) kullanacağız.

Her iki yaklaşım da farklı iş tercihlerine karşılık gelir: Yaklaşım A ("ham F1'i maksimize et") bu
dataset için **dejenere** bir sonuç veriyor ve kullanılmamalıdır; Yaklaşım B (precision koşulu
altında recall'u maksimize et) açıkça tanımlanmış, anlamlı bir iş hedefine dayanıyor.

### Critical Rules

1. Threshold seçimi **validation verisi** üzerinde yapılmalıdır (yukarıda yaptığımız gibi).
2. Threshold seçiminde **test verisi kullanılmamalıdır**.
3. Test verisindeki sonuçlara göre threshold **yeniden değiştirilmemelidir** (Section 8'de bir kez
   test edeceğiz ve sonuca göre geri dönmeyeceğiz).
4. Threshold, aynı validation verisi üzerinde seçildiği için, elde edilen validation skorunun
   (Precision ≈ 0.75) **biraz iyimser** olabileceğini unutmayın — bu nedenle nihai, bağımsız
   değerlendirme için test seti kullanılır.
5. Threshold değişimi, modelin **probability skorlarını yeniden kalibre etmez** — yalnızca hangi
   skorun "1" olarak etiketleneceğinin sınırını değiştirir.
6. **Threshold optimization** ve **probability calibration** farklı problemlerdir; birini
   yapmak diğerini otomatik olarak çözmez.

### Discussion

**"Recall artırıldığında (threshold düşürüldüğünde) False Positive sayısında nasıl bir değişim
olabilir?"**

Yukarıdaki threshold karşılaştırma tablosuna göre: threshold `0.70 → 0.30` düşürüldüğünde, Recall
`0.39 → 0.98` artarken, False Positive sayısı `6 → 641`'e çıkıyor. Bu, Recall'u artırmanın
genellikle daha fazla "yanlış alarm" (False Positive) maliyeti ile geldiğini gösteriyor.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: CHOSEN_THRESHOLD ile varsayılan 0.5 threshold'unu, validation setindeki FP ve FN sayıları
#       açısından karşılaştırın.
# TODO: Hangi threshold'un daha az False Negative ürettiğini yorumlayın.

# print(evaluate_threshold(y_val, proba_val_lr, 0.5))
# print(evaluate_threshold(y_val, proba_val_lr, CHOSEN_THRESHOLD))


---
## Section 8 — Model Comparison & Final Workshop
**Süre: 15 dakika**

### Workshop Objective

Bu LAB boyunca geliştirdiğiniz classification modellerini **aynı değerlendirme koşullarında**
karşılaştırıp, bir model + threshold seçip, bunu test verisinde **bir kez** değerlendireceksiniz.

### Hands-On — Model Comparison Table


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score


def evaluate_model(name, pipe_or_estimator, X_tr, y_tr, X_v, y_v):
    pred_train = pipe_or_estimator.predict(X_tr)
    proba_v = pipe_or_estimator.predict_proba(X_v)[:, 1]
    pred_v = (proba_v >= 0.5).astype(int)
    return {
        "Model": name,
        "Train_Accuracy": accuracy_score(y_tr, pred_train),
        "Validation_Accuracy": accuracy_score(y_v, pred_v),
        "Validation_Precision": precision_score(y_v, pred_v, zero_division=0),
        "Validation_Recall": recall_score(y_v, pred_v, zero_division=0),
        "Validation_F1": f1_score(y_v, pred_v, zero_division=0),
        "Validation_ROC_AUC": roc_auc_score(y_v, proba_v),
        "Validation_AP": average_precision_score(y_v, proba_v),
    }


comparison_rows = [
    evaluate_model("DummyClassifier(most_frequent)", baseline_pipe, X_train, y_train, X_val, y_val),
    evaluate_model("LogisticRegression", logreg_pipe, X_train, y_train, X_val, y_val),
    evaluate_model("DecisionTree(max_depth=2)", tree_models[2], X_train, y_train, X_val, y_val),
]

comparison_df = pd.DataFrame(comparison_rows).round(4)
comparison_df


### Interpretation

`DecisionTree(max_depth=2)`, varsayılan `0.5` threshold'unda **Accuracy, Precision ve F1** açısından
`LogisticRegression`'dan biraz daha iyi görünüyor. Ancak nihai model seçimi için şu noktaları da
dikkate almamız gerekir:

- **`LogisticRegression`'ın AP'si (≈ 0.839) `DecisionTree`'den (≈ 0.802) belirgin biçimde daha
  yüksek** — bu, Logistic Regression'ın tüm threshold aralığında genel olarak daha güçlü bir
  sıralama (ranking) kalitesine sahip olduğunu gösterir.
- `DecisionTree(max_depth=2)`'nin `predict_proba()` çıktısı **yalnızca 3 farklı değerden**
  oluşuyor (bir sonraki hücrede gösteriyoruz) — bu, Section 7'de yaptığımız gibi ince taneli bir
  threshold optimizasyonu yapmak için **çok kısıtlı** bir yapıdır. `LogisticRegression`'ın sürekli
  skorları, böyle bir optimizasyon için çok daha uygundur.
- `LogisticRegression`'ın katsayıları doğrudan yorumlanabilir (Section 3).

Bu nedenlerle, Section 7'de üzerinde çalıştığımız **`LogisticRegression` + `CHOSEN_THRESHOLD`**
kombinasyonunu nihai model olarak seçiyoruz.

### Hands-On — Why Threshold Tuning Needs Continuous Scores


In [ ]:
proba_val_tree2 = tree_models[2].predict_proba(X_val)[:, 1]
print("Distinct probability values produced by DecisionTree(max_depth=2):", np.unique(proba_val_tree2))
print("Distinct probability values produced by LogisticRegression        :", len(np.unique(proba_val_lr)))


### Student Tasks

1. Baseline performansını belirleyin (yukarıdaki tablo).
2. Logistic Regression sonuçlarını inceleyin (Section 3 / yukarıdaki tablo).
3. Decision Tree sonuçlarını inceleyin (Section 4 / yukarıdaki tablo).
4. Confusion Matrix karşılaştırması yapın (Section 5).
5. Validation F1 (veya başka açıkça belirlenmiş bir seçim metriği) üzerinden model adayı seçin.
6. Gerekirse validation üzerinde threshold belirleyin (Section 7 — zaten yaptık:
   `CHOSEN_THRESHOLD`).
7. Seçilen model ve threshold'u sabitleyin.
8. Test set üzerinde **tek bir** nihai değerlendirme gerçekleştirin.
9. Sonuçların sınırlamalarını açıklayın.

### Hands-On — Final Test Evaluation (ONE TIME ONLY)


In [ ]:
# This is the ONLY place in this notebook where the test set is used for evaluation.
# logreg_pipe was already fit exclusively on X_train/y_train in Section 3 — we do not refit here.
proba_test_lr = logreg_pipe.predict_proba(X_test)[:, 1]
pred_test_final = (proba_test_lr >= CHOSEN_THRESHOLD).astype(int)

final_test_accuracy = accuracy_score(y_test, pred_test_final)
final_test_precision = precision_score(y_test, pred_test_final)
final_test_recall = recall_score(y_test, pred_test_final)
final_test_f1 = f1_score(y_test, pred_test_final)
final_test_roc_auc = roc_auc_score(y_test, proba_test_lr)
final_test_ap = average_precision_score(y_test, proba_test_lr)
final_test_cm = confusion_matrix(y_test, pred_test_final, labels=[0, 1])

print(f"Threshold used : {CHOSEN_THRESHOLD}")
print(f"Test Accuracy  : {final_test_accuracy:.4f}")
print(f"Test Precision : {final_test_precision:.4f}")
print(f"Test Recall    : {final_test_recall:.4f}")
print(f"Test F1        : {final_test_f1:.4f}")
print(f"Test ROC-AUC   : {final_test_roc_auc:.4f}")
print(f"Test AP        : {final_test_ap:.4f}")
print()
print("Confusion Matrix (Test):")
print(final_test_cm)


Test sonuçları (Precision ≈ 0.729, Recall ≈ 0.628) validation sonuçlarına (Precision ≈ 0.751,
Recall ≈ 0.600) yakın büyüklük mertebesinde — modelin validation setine aşırı uydurulmadığını
(overfit edilmediğini) gösteriyor. Precision, validation'a kıyasla biraz düşük — Section 7,
Critical Rule #4'te belirttiğimiz gibi, threshold aynı validation verisi üzerinde seçildiği için
validation skorunun hafifçe iyimser olması beklenen bir durumdur.

### Important

**Test performansını gördükten sonra model veya threshold seçimini değiştirmiyoruz.** Bu sonuç,
seçilen modelin nihai, bağımsız değerlendirmesidir.

### Expected Final Summary — Classification Evaluation Summary

| Başlık | Değer |
|---|---|
| **Dataset** | E-Commerce Shipping Data (`data/ecommerce_shipping/Train.csv`, 10999 sipariş) |
| **Business Problem** | Shipment Delay Classification |
| **Target Definition** | `Reached.on.Time_Y.N`: `1` = Late Delivery (positive class), `0` = On Time |
| **Prediction Time** | Before shipment dispatch (eğitim varsayımı) |
| **Selected Features** | `Warehouse_block`, `Mode_of_Shipment`, `Product_importance` (categorical); `Cost_of_the_Product`, `Prior_purchases`, `Discount_offered`, `Weight_in_gms` (numerical) |
| **Data Splitting** | Stratified 70/15/15 (train: 7699, val: 1650, test: 1650) |
| **Baseline** | `DummyClassifier(most_frequent)` — Val Accuracy ≈ 0.597, Val Recall(1) = 1.0, Val Recall(0) = 0.0 |
| **Compared Models** | DummyClassifier, LogisticRegression, DecisionTreeClassifier (4 depth) |
| **Selected Model** | `LogisticRegression` (preprocessing pipeline dahil) |
| **Decision Threshold** | `CHOSEN_THRESHOLD` ≈ 0.53 (Precision ≥ 0.75 koşulu altında Recall maksimize edildi) |
| **Validation Findings** | Val Precision ≈ 0.751, Val Recall ≈ 0.600, Val ROC-AUC ≈ 0.725, Val AP ≈ 0.839 |
| **Final Test Results** | Test Precision ≈ 0.729, Test Recall ≈ 0.628, Test F1 ≈ 0.675, Test ROC-AUC ≈ 0.717, Test AP ≈ 0.836 |
| **Error Analysis** | FP/FN oranı threshold seçimiyle kasıtlı olarak Precision'a doğru kaydırıldı; `Discount_offered` ve `Weight_in_gms` en etkili feature'lar |
| **Dataset Limitations** | Bu, gerçek bir lojistik şirketinin verisi değildir; `Customer_rating`/`Customer_care_calls` leakage riski nedeniyle dışarıda tutuldu; feature'ların gerçek prediction-time erişilebilirliği doğrulanmadı |
| **Future Improvements** | Gerçek operasyonel maliyetlerin (FP/FN) belirlenmesi; class weighting veya probability calibration'ın araştırılması; daha fazla/daha güncel veri |

### Important

**En düşük hataya sahip (veya en yüksek Accuracy'ye sahip) modelin mutlaka production için en
uygun model olduğu iddia edilmemektedir.** Model seçimi; ranking kalitesi (AP), threshold
esnekliği, yorumlanabilirlik ve iş hedefleriyle uyum gibi birden fazla faktörün birlikte
değerlendirilmesini gerektirir.


---
## Advanced Challenges (İsteğe Bağlı)

Aşağıdaki üç challenge, ana 150 dakikalık programın **dışındadır** ve isteğe bağlıdır. Deneyimli
katılımcılar için tasarlanmıştır.

### Challenge 1 — Class Weight Comparison

`LogisticRegression`'ı `class_weight=None` ve `class_weight="balanced"` ayarlarıyla karşılaştırın.


In [ ]:
# Advanced Challenge 1 — Class Weight Comparison (optional):
# TODO: class_weight="balanced" ile yeni bir LogisticRegression pipeline'ı oluşturup eğitin.
# TODO: Validation Precision/Recall/F1'i class_weight=None (Section 3'teki logreg_pipe) ile karşılaştırın.
# TODO: Sınıf ağırlıklandırmasının HER durumda performansı iyileştireceğini VARSAYMAYIN — gerçek
#       etkisini bu dataset üzerinde ölçün ve yorumlayın.

# balanced_pipe = Pipeline(steps=[("preprocessing", preprocessor), ("model", LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE))])
# balanced_pipe.fit(X_train, y_train)
# pred_val_balanced = balanced_pipe.predict(X_val)
# print(precision_score(y_val, pred_val_balanced), recall_score(y_val, pred_val_balanced), f1_score(y_val, pred_val_balanced))


### Challenge 2 — Automated Threshold Tuning

Scikit-learn'ün `TunedThresholdClassifierCV` API'sini (mevcut sürümde bulunur) araştırın.


In [ ]:
# Advanced Challenge 2 — Automated Threshold Tuning (optional):
# TODO: sklearn.model_selection.TunedThresholdClassifierCV kullanarak, Section 7'deki manuel
#       threshold seçimine benzer bir hedefi (örn. scoring="f1" veya özel bir scorer) otomatik
#       olarak optimize edin.
# TODO: Training, validation ve test ayrımını koruyun — aynı verinin hem estimator fitting hem
#       threshold selection için kullanılmasından kaçının (örn. estimator'ı X_train üzerinde,
#       threshold tuning'i ayrı bir validation katmanında yapın).

# from sklearn.model_selection import TunedThresholdClassifierCV
# tuned = TunedThresholdClassifierCV(logreg_pipe, scoring="f1", cv="prefit")
# tuned.fit(X_val, y_val)
# print(tuned.best_threshold_)


### Challenge 3 — Error Segmentation

Seçilen modelin False Positive ve False Negative örneklerini `Warehouse_block`, `Mode_of_Shipment`
ve `Product_importance` grupları üzerinden inceleyin.


In [ ]:
# Advanced Challenge 3 — Error Segmentation (optional):
# TODO: X_val'e tahmin, gerçek değer ve hata türünü (TP/TN/FP/FN) ekleyen bir DataFrame oluşturun.
# TODO: Bu hata türlerinin Warehouse_block / Mode_of_Shipment / Product_importance bazında
#       dağılımını inceleyin.
# TODO: Grup büyüklüklerini gösterin ve küçük örneklemli gruplardaki bulguları dikkatli yorumlayın.

# val_with_errors = X_val.copy()
# val_with_errors["y_true"] = y_val.values
# val_with_errors["y_pred"] = (proba_val_lr >= CHOSEN_THRESHOLD).astype(int)
# print(val_with_errors.groupby("Warehouse_block").apply(lambda d: (d["y_true"] != d["y_pred"]).mean()))


---
## LAB06 Özeti

Bu LAB'da, LAB02-LAB05'te öğrendiğiniz bilgileri yeni bir problem türü (Binary Classification)
üzerinde kullanarak:

- Classification problemini tanımladık ve Regression'dan farkını açıkladık.
- E-Commerce Shipping verisinden, tahmin anında bilinmesi makul olan feature'ları seçtik (ve
  leakage riski taşıyan/gereksiz demografik alanları gerekçeli biçimde dışarıda tuttuk).
- Veriyi stratified olarak train/validation/test kümelerine ayırdık.
- `DummyClassifier` ile baseline performansı belirledik ve "yüksek Accuracy her zaman iyi
  performans anlamına gelmez" prensibini gerçek veriyle gösterdik.
- Logistic Regression'ın matematiğini (sigmoid, log-odds) ve `predict()`/`predict_proba()` farkını
  öğrendik.
- Decision Tree Classification'da `max_depth`'in underfitting'den overfitting'e geçişi nasıl
  tetikledigini gördük.
- Confusion Matrix'i TP/TN/FP/FN bileşenleriyle inceledik ve Accuracy, Precision, Recall, F1'i
  hem matematiksel olarak hem Scikit-learn ile hesapladık.
- ROC-AUC ve Average Precision'ı, class prevalence bağlamında yorumladık.
- Class Imbalance'ı gerçek veri (orta derecede dengesiz) ve synthetic bir örnekle (şiddetli
  dengesiz) karşılaştırarak Accuracy Paradox'u somut biçimde gösterdik.
- Decision Threshold'u validation verisinde, açıkça tanımlı bir iş hedefine göre seçtik — ve saf
  F1 maksimizasyonunun bu dataset için neden dejenere bir sonuç verdiğini gördük.
- Validation verisiyle bir model + threshold seçip, test verisini **yalnızca bir kez**, seçilen
  modelin bağımsız değerlendirmesi için kullandık.

Bu LAB'da eğittiğimiz classification pipeline'ı, bir sonraki modülde genişletilebilir bir temel
oluşturur. Bir sonraki oturumda görüşmek üzere!
